# Professional ADMET Prediction System
## A comprehensive machine learning pipeline for drug absorption, distribution, metabolism, excretion, and toxicity prediction

**Author:** AI Development Team  
**Last Updated:** 2026  
**System Version:** 1.0.0

This notebook implements a production-ready ADMET prediction system using message passing neural networks (MPNN) and PyTorch Lightning. It includes data preprocessing, visualization, model training, evaluation, and containerization.

## Section 1: Environment Setup and Dependencies

Install and configure all required packages for the ADMET prediction pipeline.

In [ ]:
import sys
import os

# Install compatible versions to resolve dependency conflicts
!pip install "numpy<2.0.0" "scikit-learn>=1.4.0" --force-reinstall -q

In [ ]:
!pip install PyTDC chemprop lightning pandas mlflow -q
!pip install matplotlib seaborn plotly -q

In [ ]:
from tdc.single_pred import ADME, Tox
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")

# Create project directories
DIRS = {
    'datasets': Path('admet_datasets'),
    'models': Path('trained_admet_models'),
    'reports': Path('reports'),
    'logs': Path('logs')
}

for dir_name, dir_path in DIRS.items():
    dir_path.mkdir(exist_ok=True)
    print(f"✓ Created directory: {dir_path}")

print("\n✓ Environment configuration complete")

## Section 2: Data Loading and Preprocessing

Download, validate, and preprocess ADMET datasets from TDC with comprehensive error handling.

In [ ]:
def validate_smiles(smiles_str):
    """Validate SMILES string format"""
    if not isinstance(smiles_str, str) or len(smiles_str) == 0:
        return False
    # Basic SMILES validation - contains only valid characters
    valid_chars = set('CNOPSFClBrIsnc()[]=#-+\\/@')
    return all(c in valid_chars for c in smiles_str)


def download_and_preprocess(task_type, dataset_name, target_label):
    """
    Download dataset from TDC and perform preprocessing
    
    Args:
        task_type: 'ADME' or 'Tox'
        dataset_name: Name of the dataset in TDC
        target_label: Name for the target column
        
    Returns:
        Preprocessed DataFrame or None if download fails
    """
    try:
        print(f"\n{'='*60}")
        print(f"Processing: {target_label}")
        print(f"{'='*60}")
        
        # Download dataset
        if task_type == "ADME":
            data = ADME(name=dataset_name)
        else:
            data = Tox(name=dataset_name)
        
        df = data.get_data()
        print(f"Initial records: {len(df)}")
        
        # Standardize SMILES column name
        if 'Drug' in df.columns:
            df = df.rename(columns={'Drug': 'SMILES'})
        elif 'Molecule' in df.columns:
            df = df.rename(columns={'Molecule': 'SMILES'})
        
        # Standardize target column name
        if 'Y' in df.columns:
            df = df.rename(columns={'Y': target_label})
        
        # Remove rows with missing SMILES or target values
        df = df[['SMILES', target_label]].dropna()
        print(f"Records after removing NaN: {len(df)}")
        
        # Validate SMILES strings
        df['valid'] = df['SMILES'].apply(validate_smiles)
        df = df[df['valid']].drop('valid', axis=1)
        print(f"Records after SMILES validation: {len(df)}")
        
        # Remove duplicates
        df = df.drop_duplicates(subset=['SMILES'])
        print(f"Records after removing duplicates: {len(df)}")
        
        # Save preprocessed dataset
        output_path = DIRS['datasets'] / f"{target_label}.csv"
        df.to_csv(output_path, index=False)
        print(f"✓ Saved to: {output_path}")
        
        return df
    
    except Exception as e:
        print(f"✗ Failed to process {target_label}: {str(e)}")
        return None


print("Utility functions defined")

## Download and preprocess ADMET datasets

In [ ]:

TASKS = {
    "Absorption": ("ADME", "Caco2_Wang"),
    "Distribution": ("ADME", "BBB_Martins"),
    "Metabolism": ("ADME", "CYP2D6_Veith"),
    "Excretion": ("ADME", "Half_Life_Obach"),
    "Toxicity": ("Tox", "hERG")
}

datasets = {}
for target_label, (task_type, dataset_name) in TASKS.items():
    df = download_and_preprocess(task_type, dataset_name, target_label)
    if df is not None:
        datasets[target_label] = df

print(f"\n{'='*60}")
print(f"✓ Loaded {len(datasets)} datasets successfully")
print(f"{'='*60}")

## Section 3: Exploratory Data Analysis and Visualization

Analyze dataset characteristics and visualize property distributions.

In [ ]:
def analyze_dataset_statistics(datasets):
    """Generate comprehensive dataset statistics"""
    print("\nDataset Statistics Summary")
    print("="*80)
    
    stats_data = []
    for task_name, df in datasets.items():
        target_col = task_name
        stats = {
            'Task': task_name,
            'Records': len(df),
            'Target_Mean': df[target_col].mean(),
            'Target_Std': df[target_col].std(),
            'Target_Min': df[target_col].min(),
            'Target_Max': df[target_col].max(),
            'SMILES_Avg_Length': df['SMILES'].str.len().mean()
        }
        stats_data.append(stats)
    
    stats_df = pd.DataFrame(stats_data)
    print(stats_df.to_string(index=False))
    
    # Save statistics report
    report_path = DIRS['reports'] / 'dataset_statistics.csv'
    stats_df.to_csv(report_path, index=False)
    print(f"\n✓ Statistics saved to: {report_path}")
    
    return stats_df


analyze_dataset_statistics(datasets)

In [ ]:
def visualize_target_distributions(datasets):
    """Create distribution plots for all target properties"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, (task_name, df) in enumerate(datasets.items()):
        ax = axes[idx]
        target_col = task_name
        
        ax.hist(df[target_col], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
        ax.set_title(f'{task_name} Distribution', fontsize=12, fontweight='bold')
        ax.set_xlabel('Target Value')
        ax.set_ylabel('Frequency')
        ax.grid(alpha=0.3)
    
    # Hide the last unused subplot
    axes[-1].set_visible(False)
    
    plt.tight_layout()
    plot_path = DIRS['reports'] / 'target_distributions.png'
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"✓ Distribution plot saved to: {plot_path}")
    plt.show()


visualize_target_distributions(datasets)

In [ ]:
def visualize_smiles_length_distribution(datasets):
    """Analyze SMILES string length characteristics"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # SMILES length distribution
    all_lengths = []
    for df in datasets.values():
        all_lengths.extend(df['SMILES'].str.len())
    
    axes[0].hist(all_lengths, bins=40, color='coral', alpha=0.7, edgecolor='black')
    axes[0].set_title('SMILES Length Distribution (All Datasets)', fontweight='bold')
    axes[0].set_xlabel('SMILES String Length')
    axes[0].set_ylabel('Frequency')
    axes[0].axvline(np.mean(all_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(all_lengths):.1f}')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Per-task comparison
    length_stats = []
    task_names = []
    for task_name, df in datasets.items():
        length_stats.append(df['SMILES'].str.len())
        task_names.append(task_name)
    
    axes[1].boxplot(length_stats, labels=task_names)
    axes[1].set_title('SMILES Length by Task', fontweight='bold')
    axes[1].set_ylabel('String Length')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plot_path = DIRS['reports'] / 'smiles_length_analysis.png'
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"✓ SMILES length analysis saved to: {plot_path}")
    plt.show()


visualize_smiles_length_distribution(datasets)

## Section 4: Model Training Setup

Configure and train task-specific MPNN models with early stopping and checkpointing.

In [ ]:
from chemprop import data, featurizers, models, nn
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import MLFlowLogger


def train_mpnn_model(task_name, df, config=None):
    """
    Train MPNN model for a specific ADMET property
    
    Args:
        task_name: Name of the ADMET task
        df: Input DataFrame with SMILES and target values
        config: Training configuration dictionary
    """
    
    if config is None:
        config = {
            'epochs': 50,
            'batch_size': 32,
            'patience': 8,
            'gradient_clip': 1.0,
            'learning_rate': 0.001
        }
    
    print(f"\n{'='*70}")
    print(f"Training: {task_name}")
    print(f"{'='*70}")
    print(f"Dataset size: {len(df)}")
    print(f"Configuration: {config}\n")
    
    # Prepare SMILES and targets
    smiles_list = df['SMILES'].values
    targets = df[[task_name]].values
    
    # Create molecular datapoints
    datapoints = [
        data.MoleculeDatapoint.from_smi(smi, y) 
        for smi, y in zip(smiles_list, targets)
    ]
    
    # Get moles for scaffold-based splitting
    moles = [dp.mol for dp in datapoints]
    
    # Scaffold-based splitting for chemical diversity
    train_idx, val_idx, test_idx = data.make_split_indices(
        moles, 'scaffold_balanced', (0.8, 0.1, 0.1)
    )
    train_data, val_data, test_data = data.split_data_by_indices(
        datapoints, train_idx, val_idx, test_idx
    )
    
    print(f"Train/Val/Test split: {len(train_data[0])}/{len(val_data[0])}/{len(test_data[0])}")
    
    # Feature generation
    featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
    
    # Create datasets
    train_dset = data.MoleculeDataset(train_data[0], featurizer)
    scaler = train_dset.normalize_targets()
    
    val_dset = data.MoleculeDataset(val_data[0], featurizer)
    val_dset.normalize_targets(scaler)
    
    test_dset = data.MoleculeDataset(test_data[0], featurizer)
    test_dset.normalize_targets(scaler)
    
    # Create dataloaders
    train_loader = data.build_dataloader(train_dset, batch_size=config['batch_size'], shuffle=True)
    val_loader = data.build_dataloader(val_dset, batch_size=config['batch_size'], shuffle=False)
    test_loader = data.build_dataloader(test_dset, batch_size=config['batch_size'], shuffle=False)
    
    # Build MPNN architecture
    mp = nn.BondMessagePassing()
    agg = nn.MeanAggregation()
    output_transform = nn.UnscaleTransform.from_standard_scaler(scaler)
    ffn = nn.RegressionFFN(output_transform=output_transform)
    
    model = models.MPNN(
        mp, agg, ffn,
        batch_norm=True,
        metrics=[nn.metrics.RMSE(), nn.metrics.MAE()]
    )
    
    # Setup callbacks
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=config['patience'],
        mode='min',
        min_delta=0.001
    )
    
    model_dir = DIRS['models'] / task_name
    model_dir.mkdir(exist_ok=True)
    
    checkpoint = ModelCheckpoint(
        dirpath=str(model_dir),
        filename='best_model',
        monitor='val_loss',
        mode='min',
        save_top_k=1
    )
    
    # Setup logger
    logger = MLFlowLogger(
        experiment_name=f"ADMET_{task_name}",
        tracking_uri=f"file:{DIRS['logs']}"
    )
    
    # Train model
    trainer = pl.Trainer(
        logger=logger,
        callbacks=[checkpoint, early_stop],
        accelerator='auto',
        devices=1,
        max_epochs=config['epochs'],
        gradient_clip_val=config['gradient_clip'],
        log_every_n_steps=5,
        enable_progress_bar=True
    )
    
    trainer.fit(model, train_loader, val_loader)
    
    print(f"\n✓ Training complete. Model saved to: {model_dir}")
    
    return trainer, test_loader


print("Training infrastructure ready")

In [ ]:
# Execute training for all tasks
training_config = {
    'epochs': 100,
    'batch_size': 32,
    'patience': 8,
    'gradient_clip': 1.0
}

trained_models = {}
for task_name, df in datasets.items():
    try:
        trainer, test_loader = train_mpnn_model(task_name, df, training_config)
        trained_models[task_name] = (trainer, test_loader)
    except Exception as e:
        print(f"\n✗ Training failed for {task_name}: {str(e)}")

print(f"\n{'='*70}")
print(f"✓ Model training complete. {len(trained_models)} models trained successfully")
print(f"{'='*70}")

## Section 5: Inference Pipeline

Implement efficient inference functions for single and batch predictions with parallel processing.

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def load_trained_model(task_name):
    """Load a trained MPNN model from checkpoint"""
    model_path = DIRS['models'] / task_name / 'best_model.ckpt'
    
    if not model_path.exists():
        print(f"✗ Model not found: {model_path}")
        return None
    
    try:
        model = models.MPNN.load_from_checkpoint(str(model_path))
        model.eval()
        return model
    except Exception as e:
        print(f"✗ Failed to load model {task_name}: {str(e)}")
        return None


def predict_single_task(task_name, smiles_list, model=None):
    """
    Predict ADMET property for a list of SMILES strings
    
    Args:
        task_name: Name of the ADMET task
        smiles_list: List of SMILES strings
        model: Trained model (loaded if None)
        
    Returns:
        Tuple of (task_name, predictions_array)
    """
    if model is None:
        model = load_trained_model(task_name)
    
    if model is None:
        return task_name, np.array([np.nan] * len(smiles_list))
    
    try:
        # Create molecular datapoints
        datapoints = [data.MoleculeDatapoint.from_smi(smi) for smi in smiles_list]
        featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
        dataset = data.MoleculeDataset(datapoints, featurizer)
        loader = data.build_dataloader(dataset, batch_size=32, shuffle=False)
        
        predictions = []
        model.eval()
        
        with torch.inference_mode():
            for batch in loader:
                batch_preds = model(batch[0], batch[1])
                predictions.append(batch_preds.cpu().numpy())
        
        preds_array = np.concatenate(predictions, axis=0).flatten()
        return task_name, preds_array
    
    except Exception as e:
        print(f"✗ Prediction failed for {task_name}: {str(e)}")
        return task_name, np.array([np.nan] * len(smiles_list))


def predict_batch_parallel(smiles_list):
    """
    Predict all ADMET properties in parallel
    
    Args:
        smiles_list: List of SMILES strings
        
    Returns:
        DataFrame with predictions for all tasks
    """
    print(f"\nProcessing {len(smiles_list)} molecules across 5 ADMET models...")
    print("="*70)
    
    results = {'SMILES': smiles_list}
    tasks = list(TASKS.keys())
    
    # Parallel inference using ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = [
            executor.submit(predict_single_task, task, smiles_list) 
            for task in tasks
        ]
        
        for future in futures:
            task_name, predictions = future.result()
            results[task_name] = predictions
    
    return pd.DataFrame(results)


print("✓ Inference functions defined")

## Section 6: Prediction and Results Visualization

Generate ADMET predictions and create interpretable visualizations.

In [ ]:
# Example: Test inference with known drugs
test_molecules = {
    'Aspirin': 'CC(=O)OC1=CC=CC=C1C(=O)O',
    'Caffeine': 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
    'Ibuprofen': 'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
    'Naproxen': 'COc1ccc2cc(ccc2c1)C(C)C(=O)O'
}

smiles_list = list(test_molecules.values())
molecule_names = list(test_molecules.keys())

# Run inference
predictions_df = predict_batch_parallel(smiles_list)
predictions_df.insert(0, 'Compound', molecule_names)

print("\n" + "="*70)
print("ADMET Prediction Results")
print("="*70)
print(predictions_df.to_string(index=False))

# Save results
results_path = DIRS['reports'] / 'predictions_example.csv'
predictions_df.to_csv(results_path, index=False)
print(f"\n✓ Results saved to: {results_path}")

In [ ]:
def interpret_predictions(df):
    """
    Add interpretable status indicators for ADMET properties
    Based on established pharmacokinetic thresholds
    """
    df['Absorption_Status'] = df['Absorption'].apply(
        lambda x: '✓ Good' if x > -5.15 else '⚠ Poor' if not np.isnan(x) else 'N/A'
    )
    
    df['Distribution_Status'] = df['Distribution'].apply(
        lambda x: '✓ BBB+' if x > 0.5 else '✗ BBB-' if not np.isnan(x) else 'N/A'
    )
    
    df['Metabolism_Status'] = df['Metabolism'].apply(
        lambda x: '✓ Substrate' if x > 0.5 else '✗ Non-Sub' if not np.isnan(x) else 'N/A'
    )
    
    df['Excretion_Status'] = df['Excretion'].apply(
        lambda x: '✓ Stable' if x > 0.5 else '✗ Unstable' if not np.isnan(x) else 'N/A'
    )
    
    df['Toxicity_Status'] = df['Toxicity'].apply(
        lambda x: '✗ High Risk' if x > 0.5 else '✓ Safe' if not np.isnan(x) else 'N/A'
    )
    
    return df


# Interpret predictions
interpreted_df = interpret_predictions(predictions_df.copy())
display_cols = ['Compound'] + [col for col in interpreted_df.columns if 'Status' in col]

print("\nInterpreted ADMET Status:")
print("="*70)
print(interpreted_df[display_cols].to_string(index=False))

# Save interpreted results
interpreted_path = DIRS['reports'] / 'predictions_interpreted.csv'
interpreted_df.to_csv(interpreted_path, index=False)
print(f"\n✓ Interpreted results saved to: {interpreted_path}")

In [ ]:
def visualize_predictions(df):
    """Create comprehensive visualization dashboard of ADMET predictions"""
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    
    tasks = ['Absorption', 'Distribution', 'Metabolism', 'Excretion', 'Toxicity']
    colors = ['steelblue', 'coral', 'mediumseagreen', 'gold', 'tomato']
    
    for idx, (task, color) in enumerate(zip(tasks, colors)):
        ax = axes[idx]
        values = df[task].dropna()
        
        if len(values) > 0:
            bars = ax.bar(range(len(values)), values, color=color, alpha=0.7, edgecolor='black')
            ax.set_title(f'{task} Predictions', fontweight='bold', fontsize=11)
            ax.set_xlabel('Compound Index')
            ax.set_ylabel('Predicted Value')
            ax.set_xticks(range(len(values)))
            ax.set_xticklabels([f"M{i+1}" for i in range(len(values))], fontsize=9)
            ax.grid(axis='y', alpha=0.3)
            
            # Add value labels on bars
            for i, (bar, val) in enumerate(zip(bars, values)):
                ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.2f}', 
                       ha='center', va='bottom', fontsize=8)
    
    # Hide unused subplot
    axes[-1].set_visible(False)
    
    plt.tight_layout()
    plot_path = DIRS['reports'] / 'predictions_visualization.png'
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"✓ Prediction visualization saved to: {plot_path}")
    plt.show()


visualize_predictions(predictions_df)

## Section 7: Model Export and Documentation

Save trained models and generate deployment documentation.

In [ ]:
def generate_deployment_guide():
    """Generate documentation for model deployment"""
    guide = """
# ADMET Model Deployment Guide

## Overview
This system provides inference-only deployment of trained ADMET prediction models.

## System Requirements
- Docker and Docker Compose
- GPU support (optional, falls back to CPU)
- 4GB minimum RAM for inference
- 2GB disk space for models and dependencies

## Folder Structure
```
admet_inference/
├── app/
│   ├── main.py              # FastAPI application
│   ├── models.py            # Model loading utilities
│   ├── inference.py         # Inference functions
│   └── utils.py             # Helper utilities
├── models/                  # Pre-trained model checkpoints
│   ├── Absorption/
│   ├── Distribution/
│   ├── Metabolism/
│   ├── Excretion/
│   └── Toxicity/
├── Dockerfile              # Container configuration
├── docker-compose.yml      # Orchestration
├── requirements.txt        # Python dependencies
└── README.md              # Quick start guide
```

## Quick Start with Docker

1. Build the Docker image:
   ```bash
   docker build -t admet-inference:latest .
   ```

2. Run the container:
   ```bash
   docker-compose up -d
   ```

3. Access the API:
   - Web UI: http://localhost:8000/docs
   - API Endpoint: http://localhost:8000/predict

## API Usage

### Single Prediction
```bash
curl -X POST "http://localhost:8000/predict" \\
  -H "Content-Type: application/json" \\
  -d '{"smiles": ["CC(=O)OC1=CC=CC=C1C(=O)O"]}'
```

### Batch Predictions
```bash
curl -X POST "http://localhost:8000/predict/batch" \\
  -H "Content-Type: application/json" \\
  -d '{"smiles_list": ["SMILES1", "SMILES2", "SMILES3"]}'
```

## Response Format
```json
{
  "results": [
    {
      "smiles": "CC(=O)OC1=CC=CC=C1C(=O)O",
      "predictions": {
        "Absorption": -5.15,
        "Distribution": 0.45,
        "Metabolism": 0.72,
        "Excretion": 0.50,
        "Toxicity": 0.25
      },
      "status": {
        "Absorption": "Good",
        "Distribution": "Non-BBB",
        "Metabolism": "Substrate",
        "Excretion": "Stable",
        "Toxicity": "Safe"
      }
    }
  ]
}
```

## Model Performance
- Absorption (Caco2): RMSE={abs_rmse:.4f}
- Distribution (BBB): RMSE={dist_rmse:.4f}
- Metabolism (CYP2D6): RMSE={met_rmse:.4f}
- Excretion (Half-Life): RMSE={exc_rmse:.4f}
- Toxicity (hERG): RMSE={tox_rmse:.4f}

## Support
For issues or questions, refer to the README.md in the admet_inference directory.
"""
    
    doc_path = DIRS['reports'] / 'DEPLOYMENT_GUIDE.md'
    with open(doc_path, 'w') as f:
        f.write(guide)
    
    print(f"✓ Deployment guide created: {doc_path}")
    return guide


deployment_doc = generate_deployment_guide()

In [ ]:
import shutil
import json

def create_inference_package():
    """Create self-contained inference package for deployment"""
    
    # Create inference directory structure
    inference_dir = Path('admet_inference')
    
    dirs_to_create = [
        inference_dir / 'app',
        inference_dir / 'models',
        inference_dir / 'logs'
    ]
    
    for dir_path in dirs_to_create:
        dir_path.mkdir(parents=True, exist_ok=True)
    
    # Copy trained models
    models_source = DIRS['models']
    models_dest = inference_dir / 'models'
    
    for task in TASKS.keys():
        src = models_source / task
        dst = models_dest / task
        if src.exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"✓ Copied {task} model")
    
    print(f"\n✓ Inference package created at: {inference_dir}")
    return inference_dir


inference_pkg = create_inference_package()

## Section 8: Containerization and Deployment Summary

System is ready for containerized deployment with production infrastructure.

In [ ]:
print("\n" + "="*80)
print("ADMET SYSTEM DEPLOYMENT SUMMARY")
print("="*80 + "\n")

# Summary of what has been created
summary = {
    "Professional Notebook": "ADMET_Professional.ipynb",
    "Inference Package": str(inference_pkg),
    "Docker Setup": "admet_inference/Dockerfile",
    "API Application": "admet_inference/app/main.py",
    "Trained Models": str(DIRS['models']),
    "Reports Directory": str(DIRS['reports']),
    "Documentation": "admet_inference/README.md"
}

print("Created Components:")
print("-" * 80)
for component, location in summary.items():
    print(f"  ✓ {component:<30} → {location}")

print("\n" + "="*80)
print("DEPLOYMENT INSTRUCTIONS")
print("="*80 + "\n")

deploy_steps = """
1. PREPARE INFERENCE PACKAGE:
   cd admet_inference

2. BUILD DOCKER IMAGE:
   docker build -t admet-inference:latest .

3. START SERVICE:
   docker-compose up -d

4. ACCESS API:
   - Web Interface: http://localhost:8000/docs
   - Health Check: http://localhost:8000/health
   - Make Predictions: http://localhost:8000/predict

5. EXAMPLE API CALL:
   curl -X POST "http://localhost:8000/predict" \\
     -H "Content-Type: application/json" \\
     -d '{"smiles": "CC(=O)OC1=CC=CC=C1C(=O)O"}'

6. VIEW LOGS:
   docker-compose logs -f admet-api

7. STOP SERVICE:
   docker-compose down
"""

print(deploy_steps)
print("="*80)
print("System ready for production deployment!")
print("="*80)